<a href="https://colab.research.google.com/github/devpatel0005/SMS-Spam-Detection/blob/main/sms_spam_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim

In [ ]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from gensim.models import Word2Vec
import tensorflow
from tensorflow import keras
from keras.models import Sequential
from keras.optimizers import Adam,SGD
from keras.regularizers import l2
from tensorflow.keras.layers import Bidirectional,LSTM,Dense,Embedding,Dropout,BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os

# --- Kaggle API setup and dataset download for Colab environment ---

# Install Kaggle API client
!pip install kaggle --quiet

# Create .kaggle directory and move kaggle.json for authentication
# The kaggle.json file is expected to be present at /content/kaggle.json as per kernel state.
!mkdir -p ~/.kaggle
!mv /content/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Download the 'SMS Spam Collection Dataset' to the current directory
# The dataset ID is 'uciml/sms-spam-collection-dataset'
!kaggle datasets download -d uciml/sms-spam-collection-dataset -p .

# Unzip the downloaded file. The zip file usually contains 'spam.csv'.
# The downloaded file is typically named after the dataset ID, so 'sms-spam-collection-dataset.zip'.
!unzip -o sms-spam-collection-dataset.zip

# Verify the contents after unzipping
print("Files in current directory after download and extraction:")
for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# --- End of Kaggle API setup ---

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df=pd.read_csv('spam.csv', encoding='latin-1')

In [ ]:
df.sample(5)

In [ ]:
df=df[['v1','v2']]

In [ ]:
df.sample(5)

In [ ]:
df.columns=['Label','message']

# text preprocessing

In [ ]:
sms=[]
for i in range(len(df)):
  text=re.sub('[^a-zA-Z]',' ',df['message'][i])
  text=text.lower()
  text=text.split()
  text=[WordNetLemmatizer().lemmatize(word) for word in text if word not in stopwords.words('english')]
  text=' '.join(text)
  sms.append([text])


In [ ]:
sms

In [ ]:
model=Word2Vec(sms,min_count=1)

- here we have created vectors for each sentence now will create training data out of it

In [ ]:
X = []
for msg in sms:
    # msg is a list containing one string, e.g., [['go jurong point crazy available bugis n great world la e buffet cine got amore wat']]
    # We need to process the string inside the list.
    word_vectors = []
    # Split the message string into individual words
    words = msg[0].split()
    for word in words:
        if word in model.wv:
            word_vectors.append(model.wv[word])

    if len(word_vectors) > 0:
        # Average the word vectors to get a sentence vector
        X.append(np.mean(word_vectors, axis=0))
    else:
        # If no words in the message are in the vocabulary, append a zero vector
        X.append(np.zeros(model.vector_size))

X = np.array(X)

# Display the shape of the generated feature matrix
print(f"Shape of feature matrix X: {X.shape}")

In [ ]:
X[0]

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['Label'])


In [ ]:
pd.DataFrame(y).value_counts().plot(kind='bar')

# Here there is class imbalance

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.1,random_state=42)

In [ ]:
smote=SMOTE()
x_train,y_train=smote.fit_resample(x_train,y_train)

In [ ]:
x_train.shape

# LSTM layers expect input with three dimensions: (number_of_samples, number_of_timesteps, number_of_features).

In [ ]:
# Reshape x_train and x_test for LSTM input
x_train_reshaped = x_train.reshape(x_train.shape[0], 1, x_train.shape[1])
x_test_reshaped = x_test.reshape(x_test.shape[0], 1, x_test.shape[1])

print(f"Reshaped x_train shape: {x_train_reshaped.shape}")
print(f"Reshaped x_test shape: {x_test_reshaped.shape}")

- he timesteps dimension is set to 1 because of how we created our X feature matrix.

We processed each message by taking all its words, finding their Word2Vec embeddings, and then averaging those word embeddings to get a single vector. This single vector represents the entire message.

Since each message is already condensed into one representative vector, when we feed it to the LSTM, it's considered a sequence of just one item – that one averaged message vector. If we had kept each word's vector separate and fed them sequentially (e.g., [[word1_vec], [word2_vec], [word3_vec]] for a three-word message), then the timesteps would be the number of words in the message. But in our current setup, each X entry is already the combined representation of a message.



In [ ]:
# Define the input dimension (vector dimension for each word)
input_dim = x_train_reshaped.shape[2]

# return_sequences=False means this is the final layer before output if we want to add another layer then make it true
model = Sequential()
model.add(Bidirectional(LSTM(128, return_sequences=True,kernel_regularizer=l2(0.001)), input_shape=(1, input_dim)))
model.add(Dropout(0.3))
model.add(BatchNormalization())
model.add(Bidirectional(LSTM(64,kernel_regularizer=l2(0.001))))
model.add(Dropout(0.3))
model.add(BatchNormalization())
model.add(Dense(1, activation='sigmoid')) # Binary classification, so 1 output unit with sigmoid activation

# Compile the model
model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
model.summary()

- input_shape=(1, input_dim) tells the Bidirectional LSTM to expect input where each sample is a sequence of 1 timestep, and that single timestep is a vector of input_dim features. here 100 is the vector dimension

In [ ]:
earlystop=EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

reducelr=ReduceLROnPlateau(
    factor=0.2,
    min_lr=0.0001,
    monitor='val_loss',
    patience=10,
)

In [ ]:
history = model.fit(x_train_reshaped, y_train, epochs=50, batch_size=32, validation_split=0.1, verbose=1,callbacks=[earlystop,reducelr])

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.show()
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])


In [ ]:
y_train_pred_probs = model.predict(x_train_reshaped)
y_train_pred = (y_train_pred_probs > 0.5).astype(int)

y_test_pred_probs = model.predict(x_test_reshaped)
y_test_pred = (y_test_pred_probs > 0.5).astype(int)


In [ ]:
from sklearn.metrics import classification_report

target_names = ['ham', 'spam']

# Generate the classification report for the training set
report_train = classification_report(y_train, y_train_pred, target_names=target_names)

print("--- Classification Report for Training Data ---")
print(report_train)

In [ ]:
from sklearn.metrics import classification_report

target_names = ['ham', 'spam']

# Generate the classification report for the test set
report_test = classification_report(y_test, y_test_pred, target_names=target_names)

print("--- Classification Report for Test Data ---")
print(report_test)

### Testing 5 Sample Messages from the Test Data

In [ ]:
import random

# Ensure test_idx is available (it should be from previous steps)
# If it's not, you might need to re-run the train_test_split cell that generates it.

# Select 5 random indices from the test set
num_samples_to_test = 5
sample_test_indices = random.sample(range(len(test_idx)), num_samples_to_test)

print("--- Sample Test Messages and Predictions ---")
for i in sample_test_indices:
    # Get the original index in the dataframe from test_idx
    original_df_index = test_idx[i]

    # Get the original message from the full dataframe 'df'
    original_message = df['message'].iloc[original_df_index]

    # Get the true label for the test set sample
    true_label = le.inverse_transform(y_test[i:i+1])[0]

    # Get the predicted label for the test set sample
    # y_test_pred is already 0 or 1. Ensure it's a 1D array for inverse_transform.
    predicted_label = le.inverse_transform(y_test_pred[i])[0]

    print(f"\nMessage: {original_message}")
    print(f"True Label: {true_label}")
    print(f"Predicted Label: {predicted_label}")
    print("------------------------------------")

# Implementing RandomForestClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the RandomForestClassifier
# You can tune hyperparameters like n_estimators, max_depth, etc.
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the classifier
print("Training RandomForestClassifier...")
rf_classifier.fit(x_train, y_train)
print("RandomForestClassifier training complete.")

In [ ]:
# Make predictions on the test set
y_pred_rf_train = rf_classifier.predict(x_train)

y_pred_rf_test = rf_classifier.predict(x_test)

# Generate classification report

print("--- RandomForestClassifier Classification Report (Train Data) ---")
print(classification_report(y_train, y_pred_rf_train, target_names=['ham', 'spam']))


print("--- RandomForestClassifier Classification Report (Test Data) ---")
print(classification_report(y_test, y_pred_rf_test, target_names=['ham', 'spam']))


# The Transformers Approach


In [ ]:
df=pd.read_csv('spam.csv', encoding='latin-1')

In [ ]:
df.sample(5)

In [ ]:
df=df[['v1','v2']]

In [ ]:
df.sample(5)

In [ ]:
x=df['v2']

In [ ]:
y=df['v1']

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2)

In [ ]:
!pip install transformers

# Steps for applying transformer
1. call the pretrained model search on the hugging face models which is better for sms spam (DistilBert)
2. Call the tokenizer, since different models have their own tokenizer
3. Convert this encodings into Dataset objects
4. Train BERT and evaluate

 # Load the Dataset

In [ ]:
import torch
from transformers import BertTokenizer,BertForSequenceClassification
#These imports are from the Hugging Face Transformers library.
#BertTokenizer is used to convert text into a format (tokens and IDs) that BERT models can understand.
#BertForSequenceClassification is a BERT model specifically fine-tuned or pre-trained for classification tasks.
from transformers import Trainer,TrainingArguments
#These are also from the Hugging Face Transformers library. Trainer is a high-level API to simplify the training and evaluation of models,
#while TrainingArguments defines the various parameters for the training process, such as learning rate, batch size, and number of epochs.
from datasets import Dataset
# It's designed to work seamlessly with models from the Hugging Face Transformers library, making it easier to prepare data for training models like BERT.

# Convert into HuggingFace Dataset

In [ ]:
train_dataset=Dataset.from_dict({
    "text":x_train.tolist(),
    "label":y_train.tolist()
})
val_dataset=Dataset.from_dict({
    "text":x_test.tolist(),
    "label":y_test.tolist()
})


# Load the Tokenizer

In [ ]:
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")

# TOkenization function

In [ ]:
def tokenize(example):
  return tokenizer(example["text"],padding="max_length",truncation=True)

In [ ]:
train_dataset=train_dataset.map(tokenize,batched=True)
val_dataset=val_dataset.map(tokenize,batched=True)


# Load the Pre-Trained BERT

In [ ]:
model=BertForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=2)

# Training Arguments

In [ ]:
training_args=TrainingArguments(
    output_dir='./results',
    learning_rate=0.01,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    logging_dir='./logs',
    logging_steps=50
)

# Trainer

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,

)

# Train

In [ ]:
trainer.train()

# Evaluate

In [ ]:
trainer.evaluate()

# Prediction Function

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)
    logits = outputs.logits

    predicted_class = torch.argmax(logits, dim=1).item()

    return "Spam" if predicted_class == 1 else "Ham"

In [ ]:
print(predict("Congratulations! You won a free iPhone"))

# Train Classification report

In [ ]:
from sklearn.metrics import classification_report

# Predict on TRAIN data
train_predictions = trainer.predict(train_dataset)

y_train_pred = train_predictions.predictions.argmax(axis=1)
y_train_true = train_predictions.label_ids

print("🔹 Train Classification Report:\n")
print(classification_report(y_train_true, y_train_pred, target_names=["Ham", "Spam"]))

# Test Classificaiotion report

In [ ]:
# Predict on TEST / VALIDATION data
val_predictions = trainer.predict(val_dataset)

y_val_pred = val_predictions.predictions.argmax(axis=1)
y_val_true = val_predictions.label_ids

print("🔹 Test Classification Report:\n")
print(classification_report(y_val_true, y_val_pred, target_names=["Ham", "Spam"]))

# Get training Logs

In [ ]:
import pandas as pd

logs = trainer.state.log_history
df_logs = pd.DataFrame(logs)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy Plot
plt.figure()
if "eval_accuracy" in df_logs:
    plt.plot(df_logs["eval_accuracy"].dropna(), label="Validation Accuracy")
plt.title("Validation Accuracy")
plt.xlabel("Steps")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Loss Plot
plt.figure()
if "loss" in df_logs:
    plt.plot(df_logs["loss"].dropna(), label="Training Loss")
if "eval_loss" in df_logs:
    plt.plot(df_logs["eval_loss"].dropna(), label="Validation Loss")

plt.title("Loss vs Steps")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.legend()
plt.show()